# layers

> Building blocks the MAX graph needs that are not in `max.experimental.nn`.
>
> Activations are channels last. Patch embedding is a 2D convolution on
> `(batch, time, mel, channels)`. Attention uses the fused `qkv` / `proj`
> names from the published encoder.


In [ ]:
#| default_exp layers


In [ ]:
#| hide
from nbdev.showdoc import *


In [ ]:
#| export
from __future__ import annotations


In [ ]:
#| export
from max.dtype import DType
from max.experimental import functional as F
from max.experimental.nn import Linear, Module
from max.experimental.tensor import Tensor
from max.graph.type import FilterLayout


`Conv2d` keeps its weight in the PyTorch layout, `(out, in, kH, kW)`, so checkpoint keys load as they are. The graph rearranges it into the RSCF filter that CPU `conv2d` takes. `permute=True` on `max.experimental.nn.Conv2d` is cuDNN-only.


In [ ]:
#| export
class Conv2d(Module):
    """2D convolution over `(batch, height, width, channels)` with stride."""

    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int | tuple[int, int],
        *,
        stride: int | tuple[int, int] = 1,
        bias: bool = True,
    ) -> None:
        kh, kw = (kernel_size, kernel_size) if isinstance(kernel_size, int) else kernel_size
        sh, sw = (stride, stride) if isinstance(stride, int) else stride
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = (kh, kw)
        self.stride = (sh, sw)
        self.weight = Tensor.zeros([out_channels, in_channels, kh, kw], dtype=DType.float32)
        self.bias = Tensor.zeros([out_channels], dtype=DType.float32) if bias else None

    def forward(self, x: Tensor) -> Tensor:
        filters = self.weight.permute([2, 3, 1, 0])
        return F.conv2d(
            x,
            filters,
            self.stride,
            (1, 1),
            (0, 0, 0, 0),
            1,
            self.bias,
            filter_layout=FilterLayout.RSCF,
        )


In [ ]:
#| export
class Attention(Module):
    """Bidirectional multi-head attention with a fused QKV projection.

    Names match the published encoder: `qkv` and `proj`. There is no mask.
    """

    def __init__(self, dim: int, num_heads: int, qkv_bias: bool = True) -> None:
        if dim % num_heads:
            raise ValueError(f"{dim} is not divisible by {num_heads} heads")
        self.heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.qkv = Linear(dim, 3 * dim, bias=qkv_bias)
        self.proj = Linear(dim, dim)

    def forward(self, hidden: Tensor) -> Tensor:
        batch, seq = hidden.shape[0], hidden.shape[1]
        qkv = self.qkv(hidden).reshape([batch, seq, 3, self.heads, self.head_dim])
        qkv = qkv.permute([2, 0, 3, 1, 4])
        query, key, value = qkv[0], qkv[1], qkv[2]
        scores = (query @ key.permute([0, 1, 3, 2])) * self.scale
        context = F.softmax(scores, axis=-1) @ value
        context = context.permute([0, 2, 1, 3]).reshape([batch, seq, self.heads * self.head_dim])
        return self.proj(context)


In [ ]:
#| export
class MLP(Module):
    """Linear, GELU, linear. Names match the published encoder: `fc1` and `fc2`."""

    def __init__(self, dim: int, hidden: int) -> None:
        self.fc1 = Linear(dim, hidden)
        self.fc2 = Linear(hidden, dim)

    def forward(self, hidden: Tensor) -> Tensor:
        return self.fc2(F.gelu(self.fc1(hidden)))
